In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j


import findspark
findspark.init()
findspark.find()

import pyspark


from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://cli.github.com/packages stable/main amd64 Packages [346 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages 

In [ ]:
import os
os.sys.path.append("/content/drive/MyDrive/Colab Notebooks/scorewarrior/features_ds_fs")

In [ ]:
import pyspark.sql.functions as F
import datetime
from features_ds_fs.base import ExtraColumn, Feature
from pyspark.sql.types import (
    IntegerType,
    StringType,
)
import pandas as pd
import numpy as np
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.stat import Correlation
from typing import List, Optional, Union
from pyspark.sql.window import Window
from dataclasses import dataclass
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, List, Tuple

from features_ds_fs.extra_cols import StartDateInTable
from features_ds_fs.base import ExtraColumn, Table
from features_ds_fs.feature_store import FS

In [ ]:
def unix_to_date(date_name):
  format = "yyyy-MM-dd HH:mm:ss"
  return F.to_timestamp((F.from_unixtime(F.col(date_name).cast(IntegerType()),format)),format)


In [ ]:
data_path = "/content/drive/MyDrive/Colab Notebooks/scorewarrior/data"

user_profile_test = (
    spark.read.csv(f"{data_path}/user_profile_test.csv", header=True, inferSchema=True)
    .select("country", "entry_point", "user_id", unix_to_date("reg_ts").alias("reg_ts"))
    )
user_profile = (
    spark.read.csv(f"{data_path}/user_profile.csv", header=True, inferSchema=True)
    .select("country", "entry_point", "user_id", unix_to_date("reg_ts").alias("reg_ts"))
    )


events = (spark.read.csv(f"{data_path}/events.csv", header=True, inferSchema=True)
                .withColumn("event_dt", unix_to_date("event_ts")))
events_test = (spark.read.csv(f"{data_path}/events_test.csv", header=True, inferSchema=True)
                .withColumn("event_dt", unix_to_date("event_ts")))

In [ ]:
(set(pd.read_csv(f"{data_path}/events_test.csv")['user_id'])
& set(pd.read_csv(f"{data_path}/events.csv")['user_id']))

set()

In [ ]:
input_events_test = (
    events_test.groupBy("user_id").agg(F.date_add(F.max("event_dt"), 1).cast(StringType()).alias("event_dt_day"))
    .join(
     user_profile_test.select("user_id", "country", "entry_point", F.date_add("reg_ts", 8).cast(StringType()).alias("reg_dt_day")),
     on = ['user_id'], how='right'
    )
    .select("user_id", "country", "entry_point", F.greatest("event_dt_day", "reg_dt_day").alias("event_dt_day"))
    .distinct()
)

In [ ]:
input_events = (
    events.groupBy("user_id").agg(F.date_add(F.max("event_dt"), 1).cast(StringType()).alias("event_dt_day"))
    .join(
     user_profile.select("user_id", "country", "entry_point", F.date_add("reg_ts", 8).cast(StringType()).alias("reg_dt_day")),
     on = ['user_id'], how='right'
    )
    .select("user_id", "country", "entry_point", F.greatest("event_dt_day", "reg_dt_day").alias("event_dt_day"))
    .distinct()
)

In [ ]:
user_profile.count() - input_events.count()

0

In [ ]:
input_events = input_events.unionByName(input_events_test)

In [ ]:
input_countries = (
    input_events
    .select("country", "event_dt_day")
    .distinct()
)
input_entries = (
    input_events_test
    .select("entry_point", "event_dt_day")
    .distinct()
)

In [ ]:
@dataclass(unsafe_hash=False, frozen=False)
class EventsTable(Table):
    """
    """

    name: Optional[str] = "EventsTable"
    query: Optional[str] = None
    client_id_col: Optional[str] = None
    date_col: Optional[str] = None
    psdf_table: Optional[pyspark.sql.DataFrame] = None
    apply_filter: bool = False
    data_path: str = "/content/drive/MyDrive/Colab Notebooks/scorewarrior/data"

    def __post_init__(self):
        events = (super().spark.read.csv(
            f"{self.data_path}/events.csv", header=True, inferSchema=True
        ).unionByName(
          super().spark.read.csv(
            f"{self.data_path}/events_test.csv", header=True, inferSchema=True

        ))
        .withColumn("event_dt_day", F.to_date(F.from_unixtime(F.col("event_ts").cast(IntegerType()),
                                                    'yyyy-MM-dd'), "yyyy-MM-dd"))
        .withColumn("event_dt_sec", F.to_timestamp(F.from_unixtime(F.col("event_ts").cast(IntegerType()),
                                                    'yyyy-MM-dd hh:mm:ss'), "yyyy-MM-dd HH:mm:ss")))
        users_profile = (
            (super().spark.read.csv(
                f"{self.data_path}/user_profile.csv", header=True, inferSchema=True
            ).select('user_id', 'country', 'entry_point')
            ).unionByName(
                super().spark.read.csv(
                f"{self.data_path}/user_profile_test.csv", header=True, inferSchema=True
            ).select('user_id', 'country', 'entry_point')
            )
        )
        self.psdf_table = events.join(users_profile, on=['user_id'], how='left')

        if self.query:
            self.psdf_table = self.psdf_table.where(self.query)

    def __hash__(self) -> int:
        return self.psdf_table.semanticHash()

    def __eq__(self, other):
        return (
            self.name,
            self.query,
            self.date_col,
            self.client_id_col,
            hash(self),
        ) == (other.name, other.query, other.date_col, other.client_id_col, hash(other))

    def __getattr__(self, attr):
        return getattr(self.psdf_table, attr)

    def __getitem__(self, item):
        return getattr(self, item)

    def filter_dates(self, date_from: datetime.date, date_to: datetime.date):
        # ничего тут не фильтруем
        pass

    def join(
        self,
        right: Union[pyspark.sql.DataFrame, "EventsTable"],
        conditions: List,
        how: str = None,
    ):
        if isinstance(right, Table):
            self.psdf_table = self.psdf_table.join(right.psdf_table, conditions, how)
        else:
            self.psdf_table = self.psdf_table.join(right, conditions, how)

    def add_columns(self, extra_columns: List[ExtraColumn]):
        for col in extra_columns:
            if col is None:
                continue

            if col.name in self.psdf_table.columns:
                continue

            self.psdf_table = self.psdf_table.withColumn(
                col.name,
                col.get(),
            )

# Агрегаты подневные

## MeanFractionDay

In [ ]:
@dataclass
class MeanFractionDay(Feature):
    """
    Фича: отношение сглаженных n_smooth_days дневная на периоде n_last_days_numerator
    Тонкости: если fs_sample_date - n_last_days_numerator будет выходить за
            дату первого появления inn в таблице то эти данные в средней учитывать не будет
    """

    name: str = "MeanFraction"
    n_last_days_numerator: int = None
    n_last_days_denumerator: int = None
    name_feature_col: str = "event_value"
    date_col:str = "event_dt_day"
    client_id_col:str = "user_id"
    events_type:list = None
    qry: Any = None
    table_query: str = None
    agg_funcs: dict = None

    def __post_init__(self):
        self.table: EventsTable = EventsTable(
            client_id_col=self.client_id_col, date_col=self.date_col, query=self.table_query
        )

        self.extra_cols: Tuple[ExtraColumn] = (
            StartDateInTable(name="start_date_entry", col_key=self.client_id_col, col_date=self.date_col),
        )

        all_events_type = [
            "battle",
            "login",
            "finish_quest",
            "payment",
        ]

        self.events_type = self.events_type or all_events_type

    def calc(self, sample_date_col: str):
        filter_event = self.isin_filter(
            column="event_name", value=self.events_type
        )

        filters_numerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        )

        first_day_filter_numerator, last_day_filter_numerator = (
            self.n_days_bound_filter(
                value=self.n_last_days_numerator,
                sample_date_col=sample_date_col,
            )
        )

        filters_denumerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_denumerator,
            sample_date_col=sample_date_col,
        )

        first_day_filter_denumerator, last_day_filter_denumerator = (
            self.n_days_bound_filter(
                value=self.n_last_days_denumerator,
                sample_date_col=sample_date_col,
            )
        )
        short_events_types_str = "_".join(self.events_type)

        n_last_days_numerator_str = (
            "ever"
            if self.n_last_days_numerator is None
            else f"{str(self.n_last_days_numerator)}days"
        )

        n_last_days_denumerator_str = (
            "ever"
            if self.n_last_days_denumerator is None
            else f"{str(self.n_last_days_denumerator)}days"
        )

        self.qry = [
            (
                (
                    F.sum(
                        F.when(
                            filters_numerator & filter_event,
                            F.col(self.name_feature_col),
                        )
                    )
                    * F.min(
                        F.datediff(
                            last_day_filter_denumerator,
                            F.greatest(
                                first_day_filter_denumerator, "start_date_entry"
                            ),
                        )
                    )
                )
                / (
                    F.sum(
                        F.when(
                            filters_denumerator & filter_event,
                            F.col(self.name_feature_col),
                        )
                    )
                    * F.min(
                        F.datediff(
                            last_day_filter_numerator,
                            F.greatest(first_day_filter_numerator, "start_date_entry"),
                        )
                    )
                )
            )
            .cast("double")
            .alias(
                f"f_{self.name}_{short_events_types_str}_prev{n_last_days_numerator_str}_to{n_last_days_denumerator_str}"
            )
        ]
        return self.qry


## MeanTrendFraction

In [ ]:
@dataclass
class MeanTrendFractionDay(Feature):
    """
    Фича: тренд сглаженных n_smooth_days дневная на периоде n_last_days_numerator
    (одинаковые периоды длинной n_smooth_days с интервалом n_last_days_denumerator-1)
    """

    name: str = "MeanTrendFraction"
    n_last_days_numerator: int = None
    n_last_days_denumerator: int = None
    name_feature_col: str = "event_value"

    client_id_col:str = "user_id"
    date_col:str = "event_dt_day"
    events_type:list = None
    qry: Any = None
    table_query: str = None
    agg_funcs: dict = None

    def __post_init__(self):
        self.table: EventsTable = EventsTable(
            client_id_col=self.client_id_col, date_col=self.date_col, query=self.table_query
        )

        self.extra_cols: Tuple[ExtraColumn] = (
            StartDateInTable(name="start_date_entry", col_key=self.client_id_col, col_date=self.date_col),
        )

        all_events_type = [
            "battle",
            "login",
            "finish_quest",
            "payment",
        ]

        self.events_type = self.events_type or all_events_type

    def calc(self, sample_date_col: str):
        filter_event = self.isin_filter(
            column="event_name", value=self.events_type
        )

        filters_numerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        )

        filters_denumerator = self.n_days_interval_filter(
            column=self.date_col,
            go_back_n_days=self.n_last_days_numerator,
            interval_len=self.n_last_days_denumerator-1,
            sample_date_col=sample_date_col,
        )
        short_events_types_str = "_".join(self.events_type)

        n_last_days_numerator_str = (
            "ever"
            if self.n_last_days_numerator is None
            else f"{str(self.n_last_days_numerator)}days"
        )

        n_last_days_denumerator_str = (
            "ever"
            if self.n_last_days_denumerator is None
            else f"{str(self.n_last_days_denumerator)}days"
        )

        self.qry = [
            (
                F.sum(
                    F.when(
                        filters_numerator & filter_event,
                        F.col(self.name_feature_col),
                    )
                )
                / F.sum(
                    F.when(
                        filters_denumerator & filter_event,
                        F.col(self.name_feature_col),
                    )
                )
            )
            .cast("double")
            .alias(
                f"f_{self.name}_{short_events_types_str}_prev{n_last_days_numerator_str}_to{n_last_days_denumerator_str}"
            )
        ]
        return self.qry


## MeanSmoothIntervalDay



In [ ]:
@dataclass
class MeanSmoothIntervalDay(Feature):
    """
    Фича: сглаженная n_smooth_days дневная на периоде n_last_days_numerator
    Тонкости: если fs_sample_date - n_last_days_numerator будет выходить за
            дату первого появления inn в таблице, то эти данные в средней учитываться не будут
            если за какой-то день после первого появления в таблице у инн нет выручки
                 - будет считаться как 0 за день и влиять на среднюю
    """

    name: str = "MeanSmoothInterval"
    n_last_days_numerator: int = None
    n_smooth_days: int = 1
    name_feature_col: str = "event_value"
    date_col:str = "event_dt_day"
    events_type:list = None
    qry: Any = None
    table_query: str = None
    agg_funcs: dict = None
    client_id_col:str = "user_id"
    def __post_init__(self):
        self.table: EventsTable = EventsTable(
            client_id_col=self.client_id_col, date_col=self.date_col, query=self.table_query
        )

        self.extra_cols: Tuple[ExtraColumn] = (
            StartDateInTable(name="start_date_entry", col_key=self.client_id_col, col_date=self.date_col),
        )

        all_events_type = [
            "battle",
            "login",
            "finish_quest",
            "payment",
        ]

        self.events_type = self.events_type or all_events_type

    def calc(self, sample_date_col: str):
        filters_numerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        ) & self.isin_filter(
            column="event_name", value=self.events_type
        )

        first_day_filter, last_day_filter = self.n_days_bound_filter(
            value=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        )

        short_events_types_str = "_".join(self.events_type)

        n_last_days_numerator_str = (
            "ever"
            if self.n_last_days_numerator is None
            else f"{str(self.n_last_days_numerator)}days"
        )

        self.qry = [
            (
                F.sum(F.when(filters_numerator, F.col(self.name_feature_col)))
                / F.min(
                    F.datediff(
                        last_day_filter,
                        F.greatest(first_day_filter, "start_date_entry"),
                    )
                    / F.lit(self.n_smooth_days)
                )
            )
            .cast("double")
            .alias(
                f"f_{self.name}_{short_events_types_str}_prev{n_last_days_numerator_str}_smooth{self.n_smooth_days}days"
            )
        ]
        return self.qry

## IntervalDay

In [ ]:
@dataclass
class IntervalDay(Feature):
    name: str = "Interval"
    n_last_days_numerator: int = None
    events_type:int = None
    name_feature_col: str = "event_value"
    date_col:str = "event_dt_day"
    client_id_col:str = 'user_id'
    qry: Any = None
    table_query: str = None
    agg_funcs: dict = field(
        default_factory=lambda: {
            "sum": F.sum,
            "max": F.max,
            "min": F.min,
            "count": F.count
        }
    )

    def __post_init__(self):
        self.table: EventsTable = EventsTable(
            client_id_col=self.client_id_col, date_col=self.date_col, query=self.table_query
        )

        all_events_type = [
            "battle",
            "login",
            "finish_quest",
            "payment",
        ]

        self.events_type = self.events_type or all_events_type

    def calc(self, sample_date_col: str):
        filters_numerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        ) & self.isin_filter(
            column="event_name", value=self.events_type
        )

        short_events_types_str = "_".join(self.events_type)
        n_last_days_numerator_str = (
            "ever"
            if self.n_last_days_numerator is None
            else f"{str(self.n_last_days_numerator)}days"
        )

        self.qry = [
            (
                self.agg_funcs[stat](
                    F.when(filters_numerator, F.col(self.name_feature_col))
                )
            )
            .cast("double")
            .alias(f"f_{self.name}_{stat}_{short_events_types_str}_prev{n_last_days_numerator_str}")
            for stat in list(self.agg_funcs.keys())
        ]
        return self.qry


## FractionDay

In [ ]:
@dataclass
class FractionDay(Feature):
    name: str = "Fraction"
    n_last_days_numerator: int = None
    n_last_days_denumerator: int = None
    events_type:int = None
    name_feature_col: str = "event_value"
    date_col:str = "event_dt_day"
    client_id_col:str = 'user_id'
    qry: Any = None
    table_query: str = None
    agg_funcs: dict = field(
        default_factory=lambda: {
            "sum": F.sum,
            "max": F.max,
            "min": F.min,
        }
    )

    def __post_init__(self):
        self.table: EventsTable = EventsTable(
            client_id_col=self.client_id_col, date_col=self.date_col, query=self.table_query
        )

        all_events_type = [
            "battle",
            "login",
            "finish_quest",
            "payment",
        ]

        self.events_type = self.events_type or all_events_type

    def calc(self, sample_date_col: str):
        filters_numerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        ) & self.isin_filter(
            column="event_name", value=self.events_type
        )

        filters_denumerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_denumerator,
            sample_date_col=sample_date_col,
        ) & self.isin_filter(
            column="event_name", value=self.events_type
        )
        short_events_types_str = "_".join(self.events_type)
        n_last_days_numerator_str = (
            "ever"
            if self.n_last_days_numerator is None
            else f"{str(self.n_last_days_numerator)}days"
        )
        n_last_days_denominator_str = (
            "ever"
            if self.n_last_days_denumerator is None
            else f"{str(self.n_last_days_denumerator)}days"
        )

        self.qry = [
            (
                self.agg_funcs[stat](
                    F.when(
                        filters_numerator, F.col(self.name_feature_col)
                    ).otherwise(0)
                )
                / self.agg_funcs[stat](
                    F.when(
                        filters_denumerator, F.col(self.name_feature_col)
                    ).otherwise(0)
                )
            )
            .cast("double")
            .alias(
                f"f_{self.name}_{stat}_{short_events_types_str}_prev{str(n_last_days_numerator_str)}_to{n_last_days_denominator_str}"
            )
            for stat in list(self.agg_funcs.keys())
        ]
        return self.qry

## TrendFractionDay

In [ ]:
@dataclass
class TrendFractionDay(Feature):
    name: str = "TrendFraction"
    n_last_days_numerator: int = None
    n_last_days_denumerator: int = None
    events_type:int = None
    name_feature_col: str = "event_value"
    date_col:str = "event_dt_day"

    client_id_col:str = 'user_id'
    qry: Any = None
    table_query: str = None
    agg_funcs: dict = field(
        default_factory=lambda: {
            "sum": F.sum,
            "max": F.max,
            "min": F.min,
        }
    )

    def __post_init__(self):
        self.table: EventsTable = EventsTable(
            client_id_col=self.client_id_col, date_col=self.date_col, query=self.table_query
        )

        all_events_type = [
            "battle",
            "login",
            "finish_quest",
            "payment",
        ]

        self.events_type = self.events_type or all_events_type

    def calc(self, sample_date_col: str):
        filters_numerator = self.n_days_filter(
            column=self.date_col,
            value=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        ) & self.isin_filter(
            column="event_name", value=self.events_type
        )

        filters_denumerator = self.n_days_interval_filter(
            column=self.date_col,
            go_back_n_days=self.n_last_days_denumerator,
            interval_len=self.n_last_days_numerator,
            sample_date_col=sample_date_col,
        ) & self.isin_filter(
            column="event_name", value=self.events_type
        )
        short_events_types_str = "_".join(self.events_type)
        n_last_days_numerator_str = (
            "ever"
            if self.n_last_days_numerator is None
            else f"{self.n_last_days_numerator}days"
        )
        n_last_days_denumerator_str = (
            "ever"
            if self.n_last_days_denumerator is None
            else f"{self.n_last_days_denumerator}days"
        )

        self.qry = [
            (
                self.agg_funcs[stat](
                    F.when(
                        filters_numerator, F.col(self.name_feature_col)
                    ).otherwise(0)
                )
                / self.agg_funcs[stat](
                    F.when(
                        filters_denumerator, F.col(self.name_feature_col)
                    ).otherwise(0)
                )
            )
            .cast("double")
            .alias(
                f"f_{self.name}_{stat}_{short_events_types_str}_prev{str(n_last_days_numerator_str)}_to{n_last_days_denumerator_str}_interval{n_last_days_numerator_str}"
            )
            for stat in list(self.agg_funcs.keys())
        ]
        return self.qry

## Тестовая сборка фичей

In [ ]:
test_input = input_events.filter("user_id == 840")
test_input.cache()
test_input.count()

1

In [ ]:
fs = FS(
    spark=spark,
    input_data=test_input.select("user_id","event_dt_day").distinct(),
    client_id_col='user_id',
    sample_date_col='event_dt_day',
)

In [ ]:
test_daily_features_list = [MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['payment']),

                            MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=2, date_col = "event_dt_day", events_type=['payment']),

                            MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),

                            MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),

                            IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['payment']),
                            IntervalDay(n_last_days_numerator=4, date_col = "event_dt_day", events_type=['payment']),
                            IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['battle']),

                            FractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),

                            TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),
                       ]

In [ ]:
test_daily_features = fs.extract(test_daily_features_list)

In [ ]:
test_daily_features.show(vertical=True)

-RECORD 0-------------------------------------------------------------------------
 fs_client_id                                                | 840                
 fs_sample_date                                              | 2025-01-09         
 f_MeanSmoothInterval_payment_prev7days_smooth1days          | 3.8426589635949813 
 f_MeanSmoothInterval_payment_prev7days_smooth2days          | 7.6853179271899625 
 f_MeanFraction_payment_prev7days_to1days                    | NULL               
 f_MeanTrendFraction_payment_prev7days_to1days               | 1.0                
 f_Interval_sum_payment_prev7days                            | 26.89861274516487  
 f_Interval_max_payment_prev7days                            | 26.89861274516487  
 f_Interval_min_payment_prev7days                            | 26.89861274516487  
 f_Interval_count_payment_prev7days                          | 1.0                
 f_Interval_sum_payment_prev4days                            | NULL               
 f_I

## Сбор фичей

In [ ]:
fs = FS(
    spark=spark,
    input_data=input_countries,
    client_id_col='country',
    sample_date_col='event_dt_day',
)

In [ ]:
country_features_list = [MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['payment'], client_id_col='country'),
                          MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['battle'], client_id_col='country'),
                          MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['login'], client_id_col='country'),
                          MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='country'),

                          MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment'], client_id_col='country'),
                          MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle'], client_id_col='country'),
                          MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login'], client_id_col='country'),
                          MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='country'),

                          IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['payment'], client_id_col='country'),
                          IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['battle'], client_id_col='country'),
                          IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['login'], client_id_col='country'),
                          IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='country'),

                          FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment'], client_id_col='country'),
                          FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle'], client_id_col='country'),
                          FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login'], client_id_col='country'),
                          FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='country'),

                          TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment'], client_id_col='country'),
                          TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle'], client_id_col='country'),
                          TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login'], client_id_col='country'),
                          TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='country'),
                       ]

In [ ]:
country_features = fs.extract(country_features_list)

In [ ]:
(country_features
 .write
 .option("header", True)
 .mode("overwrite")
 .csv(f"{data_path}/features/country_features.csv"))

In [ ]:
fs = FS(
    spark=spark,
    input_data=input_entries,
    client_id_col='entry_point',
    sample_date_col='event_dt_day',
)

In [ ]:
entry_features_list = [MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['payment'], client_id_col='entry_point'),
                      MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['battle'], client_id_col='entry_point'),
                      MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['login'], client_id_col='entry_point'),
                      MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='entry_point'),

                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment'], client_id_col='entry_point'),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle'], client_id_col='entry_point'),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login'], client_id_col='entry_point'),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='entry_point'),

                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['payment'], client_id_col='entry_point'),
                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['battle'], client_id_col='entry_point'),
                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['login'], client_id_col='entry_point'),
                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='entry_point'),

                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment'], client_id_col='entry_point'),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle'], client_id_col='entry_point'),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login'], client_id_col='entry_point'),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='entry_point'),

                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment'], client_id_col='entry_point'),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle'], client_id_col='entry_point'),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login'], client_id_col='entry_point'),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest'], client_id_col='entry_point'),
                       ]

In [ ]:
entry_features = fs.extract(entry_features_list)

In [ ]:
(entry_features
 .write
 .option("header", True)
 .mode("overwrite")
 .csv(f"{data_path}/features/entry_features.csv"))

In [ ]:
fs = FS(
    spark=spark,
    input_data=input_events.select("user_id","event_dt_day").distinct(),
    client_id_col='user_id',
    sample_date_col='event_dt_day',
)

In [ ]:
daily_features_list = [MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['payment']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['battle']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['login']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=1, date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=2, date_col = "event_dt_day", events_type=['payment']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=2, date_col = "event_dt_day", events_type=['battle']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=2, date_col = "event_dt_day", events_type=['login']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=2, date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=3, date_col = "event_dt_day", events_type=['payment']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=3, date_col = "event_dt_day", events_type=['battle']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=3, date_col = "event_dt_day", events_type=['login']),
                       MeanSmoothIntervalDay(n_last_days_numerator=7, n_smooth_days=3, date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['battle']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['login']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['payment']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['battle']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['login']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login']),
                       MeanFractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),
                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['battle']),
                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['login']),
                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['payment']),
                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['battle']),
                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['login']),
                       MeanTrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['battle']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['login']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['payment']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['battle']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['login']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['finish_quest']),

                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login']),
                       MeanTrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest']),

                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['payment']),
                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['battle']),
                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['login']),
                       IntervalDay(n_last_days_numerator=7, date_col = "event_dt_day", events_type=['finish_quest']),

                       IntervalDay(n_last_days_numerator=4, date_col = "event_dt_day", events_type=['payment']),
                       IntervalDay(n_last_days_numerator=4, date_col = "event_dt_day", events_type=['battle']),
                       IntervalDay(n_last_days_numerator=4, date_col = "event_dt_day", events_type=['login']),
                       IntervalDay(n_last_days_numerator=4, date_col = "event_dt_day", events_type=['finish_quest']),

                       IntervalDay(n_last_days_numerator=2, date_col = "event_dt_day", events_type=['payment']),
                       IntervalDay(n_last_days_numerator=2, date_col = "event_dt_day", events_type=['battle']),
                       IntervalDay(n_last_days_numerator=2, date_col = "event_dt_day", events_type=['login']),
                       IntervalDay(n_last_days_numerator=2, date_col = "event_dt_day", events_type=['finish_quest']),

                       IntervalDay(n_last_days_numerator=1, date_col = "event_dt_day", events_type=['payment']),
                       IntervalDay(n_last_days_numerator=1, date_col = "event_dt_day", events_type=['battle']),
                       IntervalDay(n_last_days_numerator=1, date_col = "event_dt_day", events_type=['login']),
                       IntervalDay(n_last_days_numerator=1, date_col = "event_dt_day", events_type=['finish_quest']),

                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['battle']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['login']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['finish_quest']),

                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['payment']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['battle']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['login']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['finish_quest']),

                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login']),
                       FractionDay(n_last_days_numerator=7, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest']),

                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),
                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['battle']),
                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['login']),
                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['finish_quest']),

                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['payment']),
                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['battle']),
                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['login']),
                       TrendFractionDay(n_last_days_numerator=7, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['finish_quest']),

                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['payment']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['battle']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['login']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=1,  date_col = "event_dt_day", events_type=['finish_quest']),

                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['payment']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['battle']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['login']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=2,  date_col = "event_dt_day", events_type=['finish_quest']),

                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['payment']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['battle']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['login']),
                       TrendFractionDay(n_last_days_numerator=6, n_last_days_denumerator=3,  date_col = "event_dt_day", events_type=['finish_quest']),
                       ]

In [ ]:
daily_features = fs.extract(daily_features_list)

In [ ]:
daily_features.cache()
daily_features.count()

5466

In [ ]:
daily_features.filter("fs_client_id == 840").select("fs_client_id", "f_Interval_sum_payment_prev7days").show()

+------------+--------------------------------+
|fs_client_id|f_Interval_sum_payment_prev7days|
+------------+--------------------------------+
|         840|               26.89861274516487|
+------------+--------------------------------+



In [ ]:
features = [col for col in daily_features.columns if col.startswith("f_")]

In [ ]:
import math
import numpy as np
import pandas as pd
from typing import List, Tuple

def prune_low_quality_features_pd(
    df: pd.DataFrame,
    exclude_cols: List[str] = None,
    missing_ratio_thresh: float = 0.5,
    min_nonnull_ratio: float = 0.2,
    unique_ratio_thresh: float = 0.01,
    std_epsilon: float = 1e-12
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    exclude_cols = set(exclude_cols or [])
    cols = [c for c in df.columns if c not in exclude_cols]
    n_rows = len(df)
    if n_rows == 0 or len(cols) == 0:
        report_df = pd.DataFrame(columns=[
            "column","dtype","n_rows","missing","missing_ratio","nonnull","nonnull_ratio",
            "distinct","unique_ratio","std","action","reasons"
        ])
        return df.copy(), report_df

    is_string = {c: pd.api.types.is_string_dtype(df[c]) for c in cols}
    is_numeric = {c: pd.api.types.is_numeric_dtype(df[c]) for c in cols}

    missing_counts = {}
    distinct_counts = {}
    std_values = {}

    for c in cols:
        s = df[c]

        miss_mask = s.isna()

        if is_string[c]:
            s_str = s.astype("string")
            empty_mask = s_str.fillna("").str.strip().eq("")
            miss_mask = miss_mask | empty_mask

        missing_counts[c] = int(miss_mask.sum())

        distinct_counts[c] = int(s[~miss_mask].nunique(dropna=True))

        if is_numeric[c]:
            s_num = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
            std_values[c] = float(s_num.std(ddof=1)) if s_num.notna().sum() > 1 else 0.0
        else:
            std_values[c] = None

    rows = []
    to_drop = []

    for c in cols:
        missing = missing_counts[c]
        distinct = distinct_counts[c]
        std = std_values[c]

        missing_ratio = missing / n_rows if n_rows > 0 else 0.0
        nonnull = n_rows - missing
        nonnull_ratio = nonnull / n_rows if n_rows > 0 else 0.0
        unique_ratio = (distinct / nonnull) if nonnull > 0 else 0.0

        reasons = []

        if missing_ratio > missing_ratio_thresh:
            reasons.append(f"high_missing:{missing_ratio:.3f}>{missing_ratio_thresh:.3f}")
        if nonnull_ratio < min_nonnull_ratio:
            reasons.append(f"low_nonnull:{nonnull_ratio:.3f}<{min_nonnull_ratio:.3f}")
        if distinct <= 1:
            reasons.append("constant_or_single_value")
        if distinct > 0 and unique_ratio <= unique_ratio_thresh:
            reasons.append(f"low_unique_ratio:{unique_ratio:.4f}<={unique_ratio_thresh:.4f}")
        if is_numeric[c]:
            if std is None or (not (std is None) and not math.isnan(std) and std <= std_epsilon):
                reasons.append(f"near_zero_std<=({std_epsilon})")

        drop_flag = len(reasons) > 0

        rows.append({
            "column": c,
            "dtype": str(df[c].dtype),
            "n_rows": n_rows,
            "missing": missing,
            "missing_ratio": round(missing_ratio, 6),
            "nonnull": nonnull,
            "nonnull_ratio": round(nonnull_ratio, 6),
            "distinct": distinct,
            "unique_ratio": round(unique_ratio, 6),
            "std": None if std is None else float(std),
            "action": "DROP" if drop_flag else "KEEP",
            "reasons": ";".join(reasons)
        })

        if drop_flag:
            to_drop.append(c)

    report_df = pd.DataFrame(rows).sort_values(
        by=["action", "missing_ratio", "unique_ratio"],
        ascending=[True, False, True]
    ).reset_index(drop=True)

    pruned_df = df.drop(columns=to_drop) if to_drop else df.copy()
    return pruned_df, report_df


In [ ]:
daily_features_pd = daily_features.toPandas()

In [ ]:

daily_features_pd

,fs_client_id,fs_sample_date,f_MeanSmoothInterval_payment_prev7days_smooth1days,f_MeanSmoothInterval_battle_prev7days_smooth1days,f_MeanSmoothInterval_login_prev7days_smooth1days,f_MeanSmoothInterval_finish_quest_prev7days_smooth1days,f_MeanSmoothInterval_payment_prev7days_smooth2days,f_MeanSmoothInterval_battle_prev7days_smooth2days,f_MeanSmoothInterval_login_prev7days_smooth2days,f_MeanSmoothInterval_finish_quest_prev7days_smooth2days,...,f_TrendFraction_min_payment_prev6days_to3days_interval6days,f_TrendFraction_sum_battle_prev6days_to3days_interval6days,f_TrendFraction_max_battle_prev6days_to3days_interval6days,f_TrendFraction_min_battle_prev6days_to3days_interval6days,f_TrendFraction_sum_login_prev6days_to3days_interval6days,f_TrendFraction_max_login_prev6days_to3days_interval6days,f_TrendFraction_min_login_prev6days_to3days_interval6days,f_TrendFraction_sum_finish_quest_prev6days_to3days_interval6days,f_TrendFraction_max_finish_quest_prev6days_to3days_interval6days,f_TrendFraction_min_finish_quest_prev6days_to3days_interval6days
0,4111,2025-01-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.0,NaN,0.000000,0.000000,NaN,0.000000,0.00,NaN
1,4420,2025-01-11,NaN,NaN,0.142857,NaN,NaN,NaN,0.285714,NaN,...,NaN,0.000000,0.0,NaN,0.006250,0.019608,NaN,0.000000,0.00,NaN
2,258,2025-01-16,NaN,0.000000,7.142857,NaN,NaN,0.000000,14.285714,NaN,...,NaN,NaN,NaN,NaN,0.470588,1.000000,NaN,NaN,NaN,NaN
3,1015,2025-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
4,2707,2025-01-16,3.383441,18.714286,69.142857,131.428571,6.766882,37.428571,138.285714,262.857143,...,NaN,1.275862,1.0,NaN,0.617261,0.202128,NaN,1.121212,1.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5461,10451,2025-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.0,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
5462,10134,2025-01-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
5463,10527,2025-01-14,NaN,NaN,3.285714,NaN,NaN,NaN,6.571429,NaN,...,NaN,NaN,NaN,NaN,7.666667,7.666667,NaN,NaN,NaN,NaN
5464,10575,2025-01-20,6.337505,11.571429,119.000000,104.285714,12.675010,23.142857,238.000000,208.571429,...,NaN,0.864407,1.0,NaN,0.942989,0.470588,NaN,2.066667,1.25,NaN


In [ ]:
exclude = ["fs_client_id", "fs_sample_date"]

pruned_df, feat_report = prune_low_quality_features_pd(
    daily_features_pd,
    exclude_cols=exclude,
    missing_ratio_thresh=0.9,
    min_nonnull_ratio=0.1,
    unique_ratio_thresh=0.01,
    std_epsilon=1e-12
)

column
dtype
n_rows
missing
missing_ratio
nonnull
nonnull_ratio
distinct
unique_ratio
std
action
reasons


In [ ]:
unuseful_feats = list(feat_report[feat_report['action'] == 'DROP']['column'])

In [ ]:
len(unuseful_feats)

87

In [ ]:
daily_features_clean = [feat for feat in features if feat not in unuseful_feats]

In [ ]:
(daily_features.select("fs_client_id", "fs_sample_date", "test",  *daily_features_clean)
 .write
 .option("header", True)
 .mode("overwrite")
 .csv(f"{data_path}/features/daily.csv"))

# Категориалки

In [ ]:
from pyspark.ml.feature import StringIndexer
features = ['entry_point', 'country']

In [ ]:

user_profile = pd.read_csv(f"{data_path}/user_profile.csv")

In [ ]:
pd.set_option("display.max_rows", 200)

In [ ]:
user_profile.groupby('country')['target'].agg(['min', 'max', 'mean', 'median', 'count']).sort_values(by='count')

,min,max,mean,median,count
country,,,,,
BB,0.000000,0.000000,0.000000,0.000000,1
AW,26.002421,26.002421,26.002421,26.002421,1
CN,103.634517,103.634517,103.634517,103.634517,1
CK,0.000000,0.000000,0.000000,0.000000,1
CF,0.000000,0.000000,0.000000,0.000000,1
CG,0.000000,0.000000,0.000000,0.000000,1
BW,0.000000,0.000000,0.000000,0.000000,1
BH,0.000000,0.000000,0.000000,0.000000,1
FO,3722.137121,3722.137121,3722.137121,3722.137121,1


In [ ]:
user_profile = pd.read_csv(f"{data_path}/user_profile.csv")
user_profile_test = pd.read_csv(f"{data_path}/user_profile_test.csv")

In [ ]:
set(user_profile_test['entry_point']) - set(user_profile['entry_point'])

set()

In [ ]:
new_country_in_test = list(set(user_profile_test['country']) - set(user_profile['country']))

In [ ]:
new_country_in_test

['SL', 'AG']

In [ ]:
user_profile = spark.read.csv(f"{data_path}/user_profile.csv", header=True).select("user_id", *features)
user_profile_test = spark.read.csv(f"{data_path}/user_profile_test.csv", header=True).select("user_id", *features)

user_profile = user_profile.unionByName(user_profile_test)

In [ ]:
user_profile_test.groupBy(["entry_point"]).count().show()

+-----------+-----+
|entry_point|count|
+-----------+-----+
|    android|  497|
|        web|    6|
|        ios|  101|
+-----------+-----+



In [ ]:
user_profile = (user_profile
                .fillna('None', subset=['country'])
                .withColumn('country',
                            F.when(F.col("country").isin(new_country_in_test), F.lit('None'))
                            .otherwise(F.col("country"))
                            )
                )

In [ ]:
indexer = StringIndexer(inputCols=features, outputCols=[col+"_ind" for col in features])
user_profile = indexer.fit(user_profile).transform(user_profile)

In [ ]:
user_profile.groupBy(["entry_point","entry_point_ind"]).count().show()

+-----------+---------------+-----+
|entry_point|entry_point_ind|count|
+-----------+---------------+-----+
|    android|            0.0| 4524|
|        ios|            1.0|  879|
|        web|            2.0|   63|
+-----------+---------------+-----+



In [ ]:
user_profile.groupBy(["country", "country_ind"]).count().show()

+-------+-----------+-----+
|country|country_ind|count|
+-------+-----------+-----+
|     HK|       94.0|    6|
|     CH|       17.0|   61|
|     TR|        6.0|  246|
|     EC|       49.0|   19|
|     IQ|       81.0|    8|
|     AZ|       30.0|   36|
|     MX|       10.0|  161|
|     SI|       91.0|    7|
|     JM|      107.0|    4|
|     BR|        1.0|  300|
|     SY|      103.0|    5|
|     VE|       57.0|   17|
|     KR|       64.0|   13|
|     LT|       73.0|   10|
|     AE|       54.0|   17|
|     TM|      123.0|    3|
|     XK|      167.0|    1|
|     SN|      164.0|    1|
|     AO|      106.0|    4|
|     EG|       24.0|   47|
+-------+-----------+-----+
only showing top 20 rows



In [ ]:
user_profile.cache().count()

5466

In [ ]:
(user_profile.select(F.col("user_id").alias("fs_client_id"), "country_ind", "entry_point_ind")
 .write
 .option("header", True)
 .mode("overwrite")
 .csv(f"{data_path}/features/category.csv"))